# Differential Equations — Session 41
## Section 9.2: Runge–Kutta Methods

**Planned length:** 90 minutes  
**Notebook type:** Student interactive lecture

### Learning objectives

Students should be able to interpret Runge–Kutta methods as weighted slope averages, implement midpoint RK2 and classical RK4, state their local and global error orders, measure convergence, compare accuracy per function evaluation, understand an embedded adaptive step, and examine absolute-stability regions.

**Edition:** Local Instructor Interactive Edition

> **Student interactive edition — local Jupyter/Cursor workflow**
>
> 1. Run the **Local notebook setup** cell below first.
> 2. Read each explanation and derivation in order.
> 3. Run simulation and visualization cells as you reach them.
> 4. At each **Classroom Checkpoint**, stop and work out your answer before class discussion continues.
> 5. This student edition intentionally contains **no instructor answer-reveal cells and no instructor solution notes**.
>
> **Tip:** During class, use `Shift + Enter` to move through the notebook one cell at a time.

In [ ]:
# Local notebook setup — run this cell first.
import importlib.util
import platform
import sys

_REQUIRED = ["numpy", "matplotlib", "scipy", "sympy", "ipywidgets"]
_missing = [name for name in _REQUIRED if importlib.util.find_spec(name) is None]

print(f"Python {sys.version.split()[0]} on {platform.system()}")
if _missing:
    print("Missing packages:", ", ".join(_missing))
    print("From the project folder, run:")
    print("python -m pip install -r requirements.txt")
else:
    print("Student notebook environment is ready.")

### Core 90-minute path

| Time | Topic |
|---:|---|
| 0–15 min | Weighted slope averages |
| 15–32 min | A second-order RK method |
| 32–55 min | Classical RK4 |
| 55–72 min | Error and cost comparison |
| 72–84 min | Adaptive step-size principle |
| 84–90 min | Stability and exit check |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sympy as sp
from scipy.integrate import solve_ivp
from scipy.optimize import root_scalar
from scipy.linalg import expm
from IPython.display import display, Markdown
try:
    from ipywidgets import interact, FloatSlider, IntSlider, Dropdown
    WIDGETS_AVAILABLE = True
except ImportError:
    WIDGETS_AVAILABLE = False
plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["axes.grid"] = True
np.set_printoptions(precision=8, suppress=True)
def euler(f, x0, y0, h, n):
    xs=x0+h*np.arange(n+1); ys=np.zeros(n+1); ys[0]=y0
    for k in range(n): ys[k+1]=ys[k]+h*f(xs[k],ys[k])
    return xs,ys
def improved_euler(f, x0, y0, h, n):
    xs=x0+h*np.arange(n+1); ys=np.zeros(n+1); ys[0]=y0
    for k in range(n):
        p=ys[k]+h*f(xs[k],ys[k])
        ys[k+1]=ys[k]+0.5*h*(f(xs[k],ys[k])+f(xs[k+1],p))
    return xs,ys
def midpoint_rk2(f, x0, y0, h, n):
    xs=x0+h*np.arange(n+1); ys=np.zeros(n+1); ys[0]=y0
    for k in range(n):
        k1=f(xs[k],ys[k]); k2=f(xs[k]+h/2,ys[k]+h*k1/2)
        ys[k+1]=ys[k]+h*k2
    return xs,ys
def rk4(f, x0, y0, h, n):
    xs=x0+h*np.arange(n+1); ys=np.zeros(n+1); ys[0]=y0
    for k in range(n):
        k1=f(xs[k],ys[k]); k2=f(xs[k]+h/2,ys[k]+h*k1/2)
        k3=f(xs[k]+h/2,ys[k]+h*k2/2); k4=f(xs[k]+h,ys[k]+h*k3)
        ys[k+1]=ys[k]+h*(k1+2*k2+2*k3+k4)/6
    return xs,ys
def rk4_system(f,t0,y0,h,n):
    ts=t0+h*np.arange(n+1); y0=np.asarray(y0,float)
    ys=np.zeros((n+1,len(y0))); ys[0]=y0
    for k in range(n):
        k1=np.asarray(f(ts[k],ys[k]))
        k2=np.asarray(f(ts[k]+h/2,ys[k]+h*k1/2))
        k3=np.asarray(f(ts[k]+h/2,ys[k]+h*k2/2))
        k4=np.asarray(f(ts[k]+h,ys[k]+h*k3))
        ys[k+1]=ys[k]+h*(k1+2*k2+2*k3+k4)/6
    return ts,ys
print("Notebook ready.")
print("Interactive widgets available:", WIDGETS_AVAILABLE)

## Formal theory reference

### Definition 9.2-A — Explicit Runge–Kutta method

A Runge–Kutta step has the form

$$
y_{n+1}=y_n+h\sum_{i=1}^m b_i k_i,
$$

where each stage $k_i$ is a slope evaluated at a selected point depending on earlier stages.

### Method 9.2-B — Midpoint RK2

$$
k_1=f(x_n,y_n),
$$

$$
k_2=f\left(x_n+\frac h2,y_n+\frac h2k_1\right),
$$

$$
y_{n+1}=y_n+hk_2.
$$

Its global error is $O(h^2)$.

### Method 9.2-C — Classical RK4

$$
k_1=f(x_n,y_n),
$$

$$
k_2=f\left(x_n+\frac h2,y_n+\frac h2k_1\right),
$$

$$
k_3=f\left(x_n+\frac h2,y_n+\frac h2k_2\right),
$$

$$
k_4=f(x_n+h,y_n+hk_3),
$$

$$
y_{n+1}
=
y_n+\frac h6(k_1+2k_2+2k_3+k_4).
$$

The local truncation error is $O(h^5)$ and global error is $O(h^4)$.

### Definition 9.2-D — Embedded adaptive pair

Two formulas of different orders share stages. Their difference estimates local error and guides step-size adjustment.

### Definition 9.2-E — Absolute stability

A method is absolutely stable at $z=h\lambda$ for the test equation $y'=\lambda y$ when its amplification factor satisfies

$$
|R(z)|<1.
$$

### Classroom Checkpoint — RK4 Step Halving

For a fourth-order method, by what approximate factor does the global error decrease when $h$ is halved?

> Pause here. Before continuing, try to explain their reasoning before continuing.

## 1. RK2 samples the slope at an interior point

In [ ]:
x0,y0,h=0.0,1.0,1.0
f=lambda x,y:y
k1=f(x0,y0)
midpoint=(x0+h/2,y0+h*k1/2)
k2=f(*midpoint)

x=np.linspace(0,h,300)
plt.plot(x,np.exp(x),label="exact")
plt.plot([x0,midpoint[0]],[y0,midpoint[1]],marker="o",label="midpoint predictor")
plt.plot(x,y0+k2*x,linestyle="--",label="line using midpoint slope")
plt.legend(); plt.title("Midpoint RK2 geometry"); plt.show()

## 2. Four RK4 stages

In [ ]:
def rk4_stage_visual(h=1.0):
    f=lambda x,y:y-x**2+1
    x0,y0=0.0,0.5
    k1=f(x0,y0)
    k2=f(x0+h/2,y0+h*k1/2)
    k3=f(x0+h/2,y0+h*k2/2)
    k4=f(x0+h,y0+h*k3)

    points=np.array([
        [x0,y0],
        [x0+h/2,y0+h*k1/2],
        [x0+h/2,y0+h*k2/2],
        [x0+h,y0+h*k3]
    ])
    plt.scatter(points[:,0],points[:,1],s=90)
    for i,(xp,yp) in enumerate(points,1):
        plt.text(xp,yp,f"  k{i}")
    plt.xlabel("x"); plt.ylabel("stage y")
    plt.title("RK4 stage locations")
    plt.show()
    print("stages:",k1,k2,k3,k4)

if WIDGETS_AVAILABLE:
    interact(rk4_stage_visual,h=FloatSlider(min=0.1,max=1.5,step=0.1,value=1.0))
else:
    rk4_stage_visual()

## 3. Compare Euler, RK2, and RK4

In [ ]:
def method_comparison(h=0.25):
    f=lambda x,y:y-x**2+1
    exact=lambda x:(x+1)**2-0.5*np.exp(x)
    final=2.0
    n=int(round(final/h)); h=final/n

    xe,ye=euler(f,0,0.5,h,n)
    x2,y2=midpoint_rk2(f,0,0.5,h,n)
    x4,y4=rk4(f,0,0.5,h,n)
    grid=np.linspace(0,final,700)

    plt.plot(grid,exact(grid),label="exact")
    plt.plot(xe,ye,marker="o",label="Euler")
    plt.plot(x2,y2,marker="s",label="RK2")
    plt.plot(x4,y4,marker="^",label="RK4")
    plt.legend(); plt.title(fr"$h={h:.4f}$"); plt.show()

    for name,value in [("Euler",ye[-1]),("RK2",y2[-1]),("RK4",y4[-1])]:
        print(name,"error =",abs(exact(final)-value))

if WIDGETS_AVAILABLE:
    interact(method_comparison,h=FloatSlider(min=0.05,max=0.5,step=0.05,value=0.25))
else:
    method_comparison()

## 4. Convergence orders

In [ ]:
f=lambda x,y:2*x*y
exact=lambda x:np.exp(x**2-1)
hs=np.array([0.1,0.05,0.025,0.0125])
methods={"Euler":euler,"RK2":midpoint_rk2,"RK4":rk4}
all_errors={}

for name,method in methods.items():
    errs=[]
    for h in hs:
        n=int(round(0.5/h))
        errs.append(abs(exact(1.5)-method(f,1,1,h,n)[1][-1]))
    all_errors[name]=np.array(errs)
    print(name,"orders:",np.log(all_errors[name][:-1]/all_errors[name][1:])/np.log(2))
    plt.loglog(hs,all_errors[name],marker="o",label=name)

plt.xlabel("h"); plt.ylabel("final error"); plt.legend(); plt.show()

## 5. Accuracy per function evaluation

Euler uses one slope evaluation per step, RK2 uses two, and RK4 uses four. A fair comparison fixes a function-evaluation budget rather than a step count.

In [ ]:
budget=80
f=lambda x,y:y-x**2+1
exact=lambda x:(x+1)**2-0.5*np.exp(x)
final=2.0

configs=[
    ("Euler",euler,budget),
    ("RK2",midpoint_rk2,budget//2),
    ("RK4",rk4,budget//4)
]
for name,method,n in configs:
    h=final/n
    value=method(f,0,0.5,h,n)[1][-1]
    print(name,"steps =",n,"error =",abs(exact(final)-value))

## 6. A simple embedded adaptive idea

Compare one RK4 step of size $h$ with two RK4 half steps. Their difference provides an error indicator.

In [ ]:
def adaptive_rk4_demo(tol=1e-5):
    f=lambda x,y:y-x**2+1
    x,y=0.0,0.5
    final=2.0
    h=0.4
    xs=[x]; ys=[y]; hs=[]

    while x < final-1e-14:
        h=min(h,final-x)
        y_big=rk4(f,x,y,h,1)[1][-1]
        y_half=rk4(f,x,y,h/2,2)[1][-1]
        err=abs(y_half-y_big)/15

        if err <= tol or h < 1e-8:
            x+=h; y=y_half
            xs.append(x); ys.append(y); hs.append(h)

        factor=2.0 if err==0 else 0.9*(tol/err)**0.2
        h*=min(2.0,max(0.2,factor))

    exact=lambda x:(x+1)**2-0.5*np.exp(x)
    plt.plot(xs,ys,marker="o",label="adaptive RK4")
    grid=np.linspace(0,final,600)
    plt.plot(grid,exact(grid),label="exact")
    plt.legend(); plt.show()

    plt.step(xs[1:],hs,where="post")
    plt.xlabel("x"); plt.ylabel("accepted h")
    plt.title("Adaptive step sizes")
    plt.show()

    print("accepted steps:",len(hs))
    print("final error:",abs(exact(final)-ys[-1]))

if WIDGETS_AVAILABLE:
    interact(adaptive_rk4_demo,
             tol=Dropdown(options=[1e-2,1e-3,1e-4,1e-5,1e-6],value=1e-5))
else:
    adaptive_rk4_demo()

## 7. Stability polynomials

For $y'=\lambda y$:

$$
R_{\text{Euler}}(z)=1+z,
$$

$$
R_{\text{RK2}}(z)=1+z+\frac{z^2}{2},
$$

$$
R_{\text{RK4}}(z)=1+z+\frac{z^2}{2}+\frac{z^3}{6}+\frac{z^4}{24}.
$$

In [ ]:
x=np.linspace(-5,2,700)
polys={
    "Euler":1+x,
    "RK2":1+x+x**2/2,
    "RK4":1+x+x**2/2+x**3/6+x**4/24
}
for name,R in polys.items():
    plt.plot(x,np.abs(R),label=name)
plt.axhline(1,linestyle="--",label="stability boundary")
plt.ylim(0,4)
plt.xlabel(r"$z=h\lambda$")
plt.ylabel(r"$|R(z)|$")
plt.legend(); plt.title("Stability along the negative real axis"); plt.show()

## Classroom Checkpoint — Exit Check

RK4 has final error approximately $1.6\times10^{-5}$ for $h=0.2$. Estimate the error for $h=0.1$.

> Pause here. Let students commit to an answer before running the next cell.